# HW3 — Part 3: Offline FrozenLake — Behavioral & Filtered Cloning
### Offline RL · Imitation Learning · Distribution Shift
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## So Unoriginal... — Offline RL on FrozenLake

**Setting:** No environment interaction during training. Learn from a fixed trajectory dataset from a weak behavior policy π_β (expected return ≈ 0.023).

| Method | Description |
|--------|-------------|
| **Behavioral Cloning (BC)** | Supervised learning on ALL (state,action) pairs |
| **Filtered Cloning (FBC)** | Train only on **successful** trajectories (return > 0) |

Filtering removes the corrupting signal from failed episodes → substantially better policy.

In [ ]:
import pickle, os, numpy as np, torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F, gymnasium as gym
from torch.utils.data import Dataset, DataLoader

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load or generate trajectories ────────────────────────────────────────
def gen_trajectories(n=100_000, seed=42):
    import random; rng=random.Random(seed); env=gym.make("FrozenLake-v1",is_slippery=True); trajs=[]
    for _ in range(n):
        s,_=env.reset(); traj=[]; done=False
        while not done:
            a=rng.choices(range(4),weights=[.2,.3,.3,.2])[0]
            ns,r,done,_,_=env.step(a); traj.append((s,a,r)); s=ns
            if len(traj)>100: break
        trajs.append(traj)
    env.close(); return trajs

if os.path.exists('trajectories.pkl'):
    with open('trajectories.pkl','rb') as f: trajectories=pickle.load(f)
    print(f"Loaded {len(trajectories):,} trajectories")
else:
    print("Generating synthetic trajectories..."); trajectories=gen_trajectories()
    print(f"Generated {len(trajectories):,} trajectories")

# Part (a): Estimate behavior policy return
est_ret=sum(sum(s[2] for s in t) for t in trajectories)/len(trajectories)
print(f"\nπ_β expected return: {est_ret:.4f}  (hint: ~0.0226)")


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
env_off=gym.make("FrozenLake-v1",is_slippery=True)
NS,NA=env_off.observation_space.n,env_off.action_space.n

class PolicyNet(nn.Module):
    def __init__(self): super().__init__(); self.logits=nn.Parameter(torch.zeros(NS,NA))
    def forward(self,s): return F.softmax(self.logits[s],dim=-1)

class TrajDS(Dataset):
    def __init__(self,d): self.d=d
    def __len__(self): return len(self.d)
    def __getitem__(self,i): s,a=self.d[i]; return torch.tensor(s,dtype=torch.long),torch.tensor(a,dtype=torch.long)

def evaluate(pol,n=20_000):
    pol.eval(); rets=[]
    for _ in range(n):
        s,_=env_off.reset(); tot=0.; done=False
        while not done:
            with torch.no_grad(): pr=pol(s).numpy(); pr=np.clip(pr,1e-8,None); pr/=pr.sum()
            s,r,done,_,_=env_off.step(np.random.choice(NA,p=pr)); tot+=r
        rets.append(tot)
    pol.train(); return float(np.mean(rets))

# ── Part (b): Behavioral Cloning ─────────────────────────────────────────
bc_data=[(s,a) for t in trajectories for s,a,_ in t]
dl=DataLoader(TrajDS(bc_data),batch_size=1024,shuffle=True)
bc_pol=PolicyNet().to(DEVICE); bc_opt=optim.Adam(bc_pol.parameters(),lr=1e-2); crit=nn.CrossEntropyLoss()
for ep in range(10):
    tl=sum((lambda loss: (bc_opt.zero_grad(),loss.backward(),bc_opt.step(),loss.item())[-1])
           (crit(bc_pol.logits[s.to(DEVICE)],a.to(DEVICE))) for s,a in dl)
    print(f"BC epoch {ep+1}/10 loss={tl/len(dl):.4f}")
print(f"BC policy return: {evaluate(bc_pol):.4f}")

# ── Part (c): Filtered Cloning ───────────────────────────────────────────
succ=[t for t in trajectories if sum(s[2] for s in t)>0]
print(f"\nSuccessful: {len(succ):,}/{len(trajectories):,}")
fbc_data=[(s,a) for t in succ for s,a,_ in t]
dl2=DataLoader(TrajDS(fbc_data),batch_size=512,shuffle=True)
fbc_pol=PolicyNet().to(DEVICE); fbc_opt=optim.Adam(fbc_pol.parameters(),lr=1e-2)
for ep in range(15):
    tl=sum((lambda loss: (fbc_opt.zero_grad(),loss.backward(),fbc_opt.step(),loss.item())[-1])
           (crit(fbc_pol.logits[s.to(DEVICE)],a.to(DEVICE))) for s,a in dl2)
    print(f"FBC epoch {ep+1}/15 loss={tl/len(dl2):.4f}")
fbc_ret=evaluate(fbc_pol)
print(f"\nSummary:  π_β={est_ret:.4f}  BC={evaluate(bc_pol):.4f}  FBC={fbc_ret:.4f}  (target ≥0.035)")
env_off.close()
